# Market Data Module

This module provides fresh market prices and margin tiers data on demand.

**NO INPUT REQUIRED** - All data is fetched directly from Snowflake.

## Functions
- `get_market_data()` - Fetch and process market prices from all sources (Ben Soliman, Marketplace, Scrapped)
- `get_margin_tiers()` - Compute margin tiers from sales data (warehouse-level, seasonally weighted)
- `get_brand_market_percentiles()` - Fetch brand-level market margin percentiles by region/brand/category
- `fill_brand_market_fallback(df, brand_percs)` - Fill NaN market columns with brand-level percentiles

## Output Columns

**Market Data (from `get_market_data()`):**
- Raw prices: `ben_soliman_price`, `final_min_price`, `final_max_price`, etc.
- Price percentiles: `minimum`, `percentile_25`, `percentile_50`, `percentile_75`, `maximum`
- Margin tiers: `below_market`, `market_min`, `market_25`, `market_50`, `market_75`, `market_max`, `above_market`

**Margin Tiers (from `get_margin_tiers()`):**
- `margin_tier_below`, `margin_tier_1`, `margin_tier_2`, `margin_tier_3`, `margin_tier_4`, `margin_tier_5`, `margin_tier_above_1`, `margin_tier_above_2`

**Brand Market Percentiles (from `get_brand_market_percentiles()`):**
- `region`, `brand`, `cat`, `num_rows`, `nmv`
- `perc_5`, `perc_15`, `perc_25`, `perc_40`, `perc_50`, `perc_60`, `perc_75`, `perc_85`, `perc_95`

## Usage
```python
%run market_data_module.ipynb

# Get market data (no input required)
df_market = get_market_data()

# Get margin tiers (no input required)
df_margin_tiers = get_margin_tiers()

# Get brand percentiles and fill fallback for SKUs without market data
df_brand_percs = get_brand_market_percentiles()
df = fill_brand_market_fallback(df, df_brand_percs)
```


In [ ]:
# =============================================================================
# IMPORTS & CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
from datetime import datetime
import pytz
import snowflake.connector
import os

# Import setup_environment_2 for credentials
import sys
sys.path.append('..')
import setup_environment_2

# Initialize environment (loads Snowflake credentials)
setup_environment_2.initialize_env()

# Cairo timezone
CAIRO_TZ = pytz.timezone('Africa/Cairo')
CAIRO_NOW = datetime.now(CAIRO_TZ)

# =============================================================================
# SNOWFLAKE CONNECTION
# =============================================================================
def query_snowflake(query):
    """Execute a query on Snowflake and return results as DataFrame."""
    con = snowflake.connector.connect(
        user=os.environ["SNOWFLAKE_USERNAME"],
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
        database=os.environ["SNOWFLAKE_DATABASE"]
    )
    try:
        cur = con.cursor()
        cur.execute("USE WAREHOUSE COMPUTE_WH")
        cur.execute(query)
        data = cur.fetchall()
        columns = [desc[0].lower() for desc in cur.description]
        df = pd.DataFrame(data, columns=columns)
        # Convert decimal.Decimal to float
        for col in df.columns:
            if df[col].dtype == object:
                try:
                    df[col] = df[col].apply(lambda x: float(x) if hasattr(x, '__float__') else x)
                except:
                    pass
        return df
    finally:
        con.close()

def get_snowflake_timezone():
    result = query_snowflake("SHOW PARAMETERS LIKE 'TIMEZONE'")
    return result['value'].iloc[0] if len(result) > 0 else "UTC"

# Get timezone for queries
TIMEZONE = get_snowflake_timezone()

# Region-Cohort mapping
REGION_COHORT_DF = pd.DataFrame({
    'region': ['Cairo', 'Giza', 'Delta West', 'Delta East', 
               'Upper Egypt', 'Upper Egypt', 'Upper Egypt', 'Upper Egypt', 'Alexandria'],
    'cohort_id': [700, 701, 703, 704, 1124, 1126, 1123, 1125, 702]
})

print(f"Market Data Module loaded at {CAIRO_NOW.strftime('%Y-%m-%d %H:%M:%S')} Cairo time")
print(f"Snowflake timezone: {TIMEZONE}")


In [ ]:
# =============================================================================
# ALL QUERIES FOR MARKET DATA MODULE
# =============================================================================
# Note: TIMEZONE is set dynamically from Snowflake in the imports cell above

# =============================================================================
# 1. BEN SOLIMAN PRICES QUERY
# =============================================================================
BEN_SOLIMAN_QUERY = f'''
WITH lower as (
    select distinct product_id, new_d*bs_price as ben_soliman_price, INJECTION_DATE
    from (
        select maxab_product_id as product_id, INJECTION_DATE, wac1, wac_p,
            (bs_price) as bs_price, diff, cu_price,
            case when p1 > 1 then child_quantity else 0 end as scheck,
            round(p1/2)*2 as p1, p2,
            case when (ROUND(p1 / scheck) * scheck) = 0 then p1 else (ROUND(p1 / scheck) * scheck) end as new_d
        from (
            select sm.*, wac1, wac_p, 
                abs((bs_price)-(wac_p*maxab_basic_unit_count))/(wac_p*maxab_basic_unit_count) as diff,
                cpc.price as cu_price, pup.child_quantity,
                round((cu_price/bs_price)) as p1, 
                round(((bs_price)/cu_price)) as p2
            from materialized_views.savvy_mapping sm 
            join finance.all_cogs f on f.product_id = sm.maxab_product_id 
                and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between f.from_Date and f.to_date
            join PACKING_UNIT_PRODUCTS pu on pu.product_id = sm.maxab_product_id and pu.IS_BASIC_UNIT = 1 
            join cohort_product_packing_units cpc on cpc.PRODUCT_PACKING_UNIT_ID = pu.id and cohort_id = 700 
            join packing_unit_products pup on pup.product_id = sm.maxab_product_id and pup.is_basic_unit = 1  
            where bs_price is not null 
                and INJECTION_DATE::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 5 
                and diff > 0.3 and p1 > 1
        )
    )
    qualify max(INJECTION_DATE) over(partition by product_id) = INJECTION_DATE
),

m_bs as (
    select z.* from (
        select maxab_product_id as product_id, avg(bs_final_price) as ben_soliman_price, INJECTION_DATE
        from (
            select *, row_number() over(partition by maxab_product_id order by diff) as rnk_2 
            from (
                select *, (bs_final_price-wac_p)/wac_p as diff_2 
                from (
                    select *, bs_price/maxab_basic_unit_count as bs_final_price 
                    from (
                        select *, row_number() over(partition by maxab_product_id, maxab_pu order by diff) as rnk 
                        from (
                            select *, max(INJECTION_DATE::date) over(partition by maxab_product_id, maxab_pu) as max_date
                            from (
                                select sm.*, wac1, wac_p, 
                                    abs(bs_price-(wac_p*maxab_basic_unit_count))/(wac_p*maxab_basic_unit_count) as diff 
                                from materialized_views.savvy_mapping sm 
                                join finance.all_cogs f on f.product_id = sm.maxab_product_id 
                                    and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between f.from_Date and f.to_date
                                where bs_price is not null 
                                    and INJECTION_DATE::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 5 
                                    and diff < 0.3
                            )
                            qualify max_date = INJECTION_DATE
                        ) qualify rnk = 1 
                    )
                ) where diff_2 between -0.5 and 0.5 
            ) qualify rnk_2 = 1 
        ) group by all
    ) z 
    join finance.all_cogs f on f.product_id = z.product_id 
        and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between f.from_Date and f.to_date
    where ben_soliman_price between f.wac_p*0.8 and f.wac_p*1.3
)
select product_id,avg(ben_soliman_price) as ben_soliman_price
from (
select product_id,ben_soliman_price,INJECTION_DATE
from (
    select * from (
        select *,1 as prio from m_bs 
        union all
        select *, 2 as prio from lower
    )
    qualify max(INJECTION_DATE) over(partition by product_id) = INJECTION_DATE
)
qualify prio = min(prio)over(partition by product_id)
)
group by all
'''

# =============================================================================
# 2. MARKETPLACE PRICES QUERY (with region fallback)
# =============================================================================
MARKETPLACE_PRICES_QUERY = f'''
WITH MP as (
    select region, product_id,
        min(min_price) as min_price, min(max_price) as max_price,
        min(mod_price) as mod_price, min(true_min) as true_min, min(true_max) as true_max
    from (
        select mp.region, mp.product_id, mp.pu_id,
            min_price/BASIC_UNIT_COUNT as min_price,
            max_price/BASIC_UNIT_COUNT as max_price,
            mod_price/BASIC_UNIT_COUNT as mod_price,
            TRUE_MIN_PRICE/BASIC_UNIT_COUNT as true_min,
            TRUE_MAX_PRICE/BASIC_UNIT_COUNT as true_max
        from materialized_views.marketplace_prices mp 
        join packing_unit_products pup on pup.product_id = mp.product_id and pup.packing_unit_id = mp.pu_id
        join finance.all_cogs f on f.product_id = mp.product_id 
            and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between f.from_date and f.to_date
        where least(min_price, mod_price) between wac_p*0.9 and wac_p*1.3 
    )
    group by all 
),

region_mapping AS (
    SELECT * FROM (VALUES
        ('Delta East', 'Delta West'), ('Delta West', 'Delta East'),
        ('Alexandria', 'Cairo'), ('Alexandria', 'Giza'),
        ('Upper Egypt', 'Cairo'), ('Upper Egypt', 'Giza'),
        ('Cairo', 'Giza'), ('Giza', 'Cairo'),
        ('Delta West', 'Cairo'), ('Delta East', 'Cairo'),
        ('Delta West', 'Giza'), ('Delta East', 'Giza')
    ) AS region_mapping(region, fallback_region)
),

all_regions as (
    SELECT * FROM (VALUES
        ('Cairo'), ('Giza'), ('Delta West'), ('Delta East'), ('Upper Egypt'), ('Alexandria')
    ) AS x(region)
),

full_data as (
    select products.id as product_id, ar.region
    from products, all_regions ar
    where activation = 'true'
)

select region, product_id,
    min(final_min_price) as final_min_price, 
    min(final_max_price) as final_max_price,
    min(final_mod_price) as final_mod_price, 
    min(final_true_min) as final_true_min,
    min(final_true_max) as final_true_max
from (
    SELECT distinct w.region, w.product_id,
        COALESCE(m1.min_price, m2.min_price) AS final_min_price,
        COALESCE(m1.max_price, m2.max_price) AS final_max_price,
        COALESCE(m1.mod_price, m2.mod_price) AS final_mod_price,
        COALESCE(m1.true_min, m2.true_min) AS final_true_min,
        COALESCE(m1.true_max, m2.true_max) AS final_true_max
    FROM full_data w
    LEFT JOIN MP m1 ON w.region = m1.region and w.product_id = m1.product_id
    LEFT JOIN region_mapping rm ON w.region = rm.region
    LEFT JOIN MP m2 ON rm.fallback_region = m2.region AND w.product_id = m2.product_id
)
where final_min_price is not null 
group by all
'''

# =============================================================================
# 3. SCRAPPED DATA QUERY (Competitor prices from scraping)
# =============================================================================
SCRAPPED_QUERY = f'''
WITH latest_per_sku AS (
    SELECT product_name_ar,
           MAX(created_at::date) AS max_date
    FROM materialized_views.raw_scraped_data
    WHERE created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 4
    GROUP BY product_name_ar
),
main_prices AS (
    SELECT DISTINCT r.*
    FROM materialized_views.raw_scraped_data r
    JOIN latest_per_sku l
      ON r.product_name_ar = l.product_name_ar
     AND r.created_at::date = l.max_date
),
cleaned_data AS (
    SELECT query_id AS product_id,
           mp.product_name_ar,
           price AS comp_prices,
           CASE coalesce(region, area)
               WHEN 'Giza' THEN 'Cairo'
               WHEN 'Alex' THEN 'Alexandria'
               ELSE coalesce(region, area)
           END AS region,
           source_app,
           supplier
    FROM main_prices mp
    JOIN materialized_views.competitors_mapping_fixed cm 
      ON cm.match = mp.product_name_ar
),
matched_prices AS (
    SELECT *,
           comp_prices / BASIC_UNIT_COUNT AS final_price
    FROM (
        SELECT *,
               MIN(diff) OVER (PARTITION BY product_id, coalesce(supplier, source_app), region) AS min_diff
        FROM (
            SELECT cd.*,
                   wac1,
                   pup.PACKING_UNIT_id,
                   BASIC_UNIT_COUNT,
                   wac1 * BASIC_UNIT_COUNT AS pu_price,
                   ABS(comp_prices - pu_price) / pu_price AS diff
            FROM cleaned_data cd
            JOIN finance.all_cogs f 
              ON f.product_id = cd.product_id
             AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN from_date AND to_date
            JOIN PACKING_UNIT_PRODUCTs pup 
              ON pup.product_id = cd.product_id
             AND diff < 0.3
        )
        QUALIFY diff = min_diff
    )
),
target_regions AS (
    SELECT 'Cairo' AS region
    UNION ALL SELECT 'Giza'
    UNION ALL SELECT 'Alexandria'
    UNION ALL SELECT 'Delta East'
    UNION ALL SELECT 'Delta West'
    UNION ALL SELECT 'Upper Egypt'
),
product_suppliers AS (
    SELECT DISTINCT product_id, product_name_ar, coalesce(supplier, source_app) AS supplier_key,
           PACKING_UNIT_id, BASIC_UNIT_COUNT, wac1
    FROM matched_prices
),
all_combos AS (
    SELECT ps.*, tr.region AS target_region
    FROM product_suppliers ps
    CROSS JOIN target_regions tr
),
fallback_prices AS (
    SELECT ac.product_id,
           ac.product_name_ar,
           ac.supplier_key,
           ac.target_region,
           ac.PACKING_UNIT_id,
           ac.BASIC_UNIT_COUNT,
           ac.wac1,
           mp.region AS source_region,
           mp.comp_prices,
           mp.final_price,
           mp.source_app,
           mp.supplier,
           CASE
               WHEN mp.region = ac.target_region THEN 1
               WHEN ac.target_region = 'Giza' AND mp.region = 'Cairo' THEN 1
               WHEN mp.region = 'Cairo' AND ac.target_region NOT IN ('Cairo', 'Giza') THEN 2
               WHEN ac.target_region IN ('Delta East', 'Delta West', 'Alexandria')
                    AND mp.region IN ('Delta East', 'Delta West', 'Alexandria')
                    AND mp.region != ac.target_region THEN 3
               ELSE NULL
           END AS priority
    FROM all_combos ac
    JOIN matched_prices mp
      ON mp.product_id = ac.product_id
     AND coalesce(mp.supplier, mp.source_app) = ac.supplier_key
    WHERE CASE
              WHEN ac.target_region IN ('Cairo', 'Giza') 
                   THEN mp.region = 'Cairo'
              WHEN ac.target_region = 'Upper Egypt' 
                   THEN mp.region IN ('Upper Egypt', 'Cairo')
              WHEN ac.target_region IN ('Delta East', 'Delta West', 'Alexandria')
                   THEN mp.region IN ('Cairo', 'Delta East', 'Delta West', 'Alexandria')
              ELSE FALSE
          END
)
SELECT product_id,
       region,
       MIN(final_price) AS min_scrapped,
       PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY final_price) AS scrapped25,
       PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY final_price) AS scrapped50,
       PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY final_price) AS scrapped75,
       MAX(final_price) AS max_scrapped
FROM (
    SELECT product_id,
           product_name_ar,
           target_region AS region,
           source_region,
           priority,
           comp_prices,
           final_price,
           source_app,
           supplier,
           supplier_key,
           PACKING_UNIT_id,
           BASIC_UNIT_COUNT,
           wac1
    FROM fallback_prices
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_id, supplier_key, target_region
        ORDER BY priority ASC
    ) = 1
)
GROUP BY ALL
'''

# =============================================================================
# 4. PRODUCT GROUPS QUERY
# =============================================================================
GROUPS_QUERY = '''
SELECT * FROM materialized_views.sku_commercial_groups_pp
'''

# =============================================================================
# 5. SALES DATA QUERY (for NMV weighting in group processing)
# =============================================================================
SALES_QUERY = f'''
SELECT DISTINCT cpc.cohort_id, pso.product_id,
    CONCAT(products.name_ar,' ',products.size,' ',product_units.name_ar) as sku,
    brands.name_ar as brand, categories.name_ar as cat,
    sum(pso.total_price) as nmv
FROM product_sales_order pso
JOIN sales_orders so ON so.id = pso.sales_order_id
JOIN COHORT_PRICING_CHANGES cpc ON cpc.id = pso.COHORT_PRICING_CHANGE_id
JOIN products ON products.id = pso.product_id
JOIN brands ON products.brand_id = brands.id 
JOIN categories ON products.category_id = categories.id
JOIN product_units ON product_units.id = products.unit_id 
WHERE so.created_at::date BETWEEN CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 120 
    AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 1 
    AND so.sales_order_status_id NOT IN (7, 12)
    AND so.channel IN ('telesales', 'retailer')
    AND pso.purchased_item_count <> 0
    AND cpc.cohort_id IN (700,701,702,703,704,1123,1124,1125,1126)
GROUP BY ALL
'''

# =============================================================================
# 6. MARGIN STATS QUERY (STD and average margins)
# =============================================================================
MARGIN_STATS_QUERY = f'''
select product_id, cohort_id, 
    (0.6*product_std) + (0.3*brand_std) + (0.1*cat_std) as std, 
    avg_margin
from (
    select product_id, cohort_id, 
        stddev(product_margin) as product_std, 
        stddev(brand_margin) as brand_std,
        stddev(cat_margin) as cat_std, 
        avg(product_margin) as avg_margin
    from (
        select distinct product_id, order_date, cohort_id,
            (nmv-cogs_p)/nmv as product_margin, 
            (brand_nmv-brand_cogs)/brand_nmv as brand_margin,
            (cat_nmv-cat_cogs)/cat_nmv as cat_margin
        from (
            SELECT DISTINCT so.created_at::date as order_date, cpc.cohort_id, pso.product_id,
                brands.name_ar as brand, categories.name_ar as cat,
                sum(COALESCE(f.wac_p,0) * pso.purchased_item_count * pso.basic_unit_count) as cogs_p,
                sum(pso.total_price) as nmv,
                sum(nmv) over(partition by order_date, cat, brand) as brand_nmv,
                sum(cogs_p) over(partition by order_date, cat, brand) as brand_cogs,
                sum(nmv) over(partition by order_date, cat) as cat_nmv,
                sum(cogs_p) over(partition by order_date, cat) as cat_cogs
            FROM product_sales_order pso
            JOIN sales_orders so ON so.id = pso.sales_order_id   
            JOIN COHORT_PRICING_CHANGES cpc on cpc.id = pso.cohort_pricing_change_id
            JOIN products on products.id = pso.product_id
            JOIN brands on products.brand_id = brands.id 
            JOIN categories ON products.category_id = categories.id
            JOIN finance.all_cogs f ON f.product_id = pso.product_id
                AND f.from_date::date <= so.created_at::date AND f.to_date::date > so.created_at::date
            WHERE so.created_at::date between 
                date_trunc('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 120) 
                and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date
                AND so.sales_order_status_id not in (7,12)
                AND so.channel IN ('telesales','retailer')
                AND pso.purchased_item_count <> 0
            GROUP BY ALL
        )
    ) group by all 
)
'''

# =============================================================================
# 7. TARGET MARGINS QUERY
# =============================================================================
TARGET_MARGINS_QUERY = f'''
WITH cat_brand_target as (
    SELECT DISTINCT cat, brand, margin as target_bm
    FROM performance.commercial_targets cplan
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date) 
        THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date)
        ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
),
cat_target as (
    select cat, sum(target_bm * (target_nmv/cat_total)) as cat_target_margin
    from (
        select *, sum(target_nmv) over(partition by cat) as cat_total
        from (
            select cat, brand, avg(target_bm) as target_bm, sum(target_nmv) as target_nmv
            from (
                SELECT DISTINCT date, city as region, cat, brand, margin as target_bm, nmv as target_nmv
                FROM performance.commercial_targets cplan
                QUALIFY CASE 
                    WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date) 
                    THEN DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date)
                    ELSE DATE_TRUNC('month', CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - INTERVAL '1 month') 
                END = DATE_TRUNC('month', date)
            ) group by all
        )
    ) group by all 
)
SELECT DISTINCT cbt.cat, cbt.brand, cbt.target_bm, ct.cat_target_margin
FROM cat_brand_target cbt
LEFT JOIN cat_target ct ON ct.cat = cbt.cat
'''

# =============================================================================
# 8. PRODUCT BASE QUERY (WAC data)
# =============================================================================
PRODUCT_BASE_QUERY = f'''
SELECT DISTINCT
    CASE 
        WHEN cohort_id IN (700, 695) THEN 'Cairo'
        WHEN cohort_id IN (701) THEN 'Giza'
        WHEN cohort_id IN (704, 698) THEN 'Delta East'
        WHEN cohort_id IN (703, 697) THEN 'Delta West'
        WHEN cohort_id IN (696, 1123, 1124, 1125, 1126) THEN 'Upper Egypt'
        WHEN cohort_id IN (702, 699) THEN 'Alexandria'
    END AS region,
    cohort_id,
    f.product_id,
    brands.name_ar as brand,
    categories.name_ar as cat,
    f.wac1,
    f.wac_p
FROM finance.all_cogs f
JOIN products ON products.id = f.product_id
JOIN brands ON products.brand_id = brands.id
JOIN categories ON products.category_id = categories.id
CROSS JOIN (
    SELECT DISTINCT cohort_id 
    FROM COHORT_PRICING_CHANGES 
    WHERE cohort_id IN (700,701,702,703,704,1123,1124,1125,1126)
) cohorts
WHERE CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
    AND products.activation = 'true'
'''

# =============================================================================
# 10. WAREHOUSE TO COHORT MAPPING QUERY
# =============================================================================
WAREHOUSE_COHORT_MAPPING_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
)
SELECT DISTINCT
    COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
    cpc.cohort_id
FROM product_sales_order pso
LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
JOIN sales_orders so ON so.id = pso.sales_order_id
JOIN COHORT_PRICING_CHANGES cpc ON cpc.id = pso.COHORT_PRICING_CHANGE_id
WHERE so.created_at::DATE >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 120
    AND so.sales_order_status_id NOT IN (7, 12)
    AND cpc.cohort_id IN (700, 701, 702, 703, 704, 1123, 1124, 1125, 1126)
'''
MARGIN_BOUNDARIES_QUERY = f'''
WITH parent_whs AS (
    SELECT * FROM (VALUES (236,343),(1,467),(962,343)) x(parent_id,child_id)
),
all_daily_margins AS (
    SELECT
        so.created_at::DATE AS order_date,
        COALESCE(pw.parent_id, pso.warehouse_id) AS warehouse_id,
        pso.product_id,
        SUM(pso.total_price) AS nmv,
        SUM(COALESCE(f.wac_p, 0) * pso.purchased_item_count * pso.basic_unit_count) AS cogs,
        DATEDIFF('day', so.created_at::DATE,
            CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE
        ) AS days_ago,
        EXTRACT(QUARTER FROM so.created_at::DATE) AS order_quarter
    FROM product_sales_order pso
    LEFT JOIN parent_whs pw ON pw.child_id = pso.warehouse_id
    JOIN sales_orders so
        ON so.id = pso.sales_order_id
    JOIN COHORT_PRICING_CHANGES cpc
        ON cpc.id = pso.COHORT_PRICING_CHANGE_id
    JOIN finance.all_cogs f
        ON f.product_id = pso.product_id
        AND f.from_date::DATE <= so.created_at::DATE
        AND f.to_date::DATE  >  so.created_at::DATE
    WHERE so.created_at::DATE BETWEEN
            DATEADD('year', -1,
                CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
            AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE - 1
        AND so.sales_order_status_id NOT IN (7, 12)
        AND so.channel IN ('telesales', 'retailer')
        AND pso.purchased_item_count <> 0
        AND cpc.cohort_id IN (700, 701, 702, 703, 704, 1123, 1124, 1125, 1126)
    GROUP BY ALL
    HAVING nmv > 0
),

daily_with_margin AS (
    SELECT
        *,
        (nmv - cogs) / NULLIF(nmv, 0) AS daily_margin,
        nmv - cogs AS gross_profit
    FROM all_daily_margins
),

iqr_stats AS (
    SELECT
        product_id,
        warehouse_id,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY daily_margin) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY daily_margin) AS q3
    FROM daily_with_margin
    GROUP BY product_id, warehouse_id
),

iqr_cleaned AS (
    SELECT dm.*
    FROM daily_with_margin dm
    JOIN iqr_stats iq
        ON dm.product_id = iq.product_id
        AND dm.warehouse_id = iq.warehouse_id
    WHERE dm.daily_margin
        BETWEEN iq.q1 - 1.5 * (iq.q3 - iq.q1)
            AND iq.q3 + 1.5 * (iq.q3 - iq.q1)
),


-- =========================================================================
-- BOUNDARIES TRACK: Same quarter from the past year, weighted percentiles
-- =========================================================================

same_quarter_data AS (
    SELECT *
    FROM iqr_cleaned
    WHERE order_quarter = EXTRACT(QUARTER FROM
        CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::DATE)
),

boundary_with_weights AS (
    SELECT
        *,
        EXP(-0.023 * days_ago) AS time_weight
    FROM same_quarter_data
),

boundary_ordered AS (
    SELECT
        product_id,
        warehouse_id,
        daily_margin,
        time_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
            ORDER BY daily_margin
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS total_weight,
        COUNT(*) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS data_points
    FROM boundary_with_weights
),

quarter_percentiles AS (
    SELECT
        product_id,
        warehouse_id,
        MIN(CASE WHEN cum_weight >= total_weight * 0.10 THEN daily_margin END) AS MIN_BOUNDARY,
        MIN(CASE WHEN cum_weight >= total_weight * 0.50 THEN daily_margin END) AS MEDIAN_BM,
        MIN(CASE WHEN cum_weight >= total_weight * 0.90 THEN daily_margin END) AS MAX_BOUNDARY
    FROM boundary_ordered
    WHERE data_points >= 5
    GROUP BY product_id, warehouse_id
),


-- =========================================================================
-- BOUNDARIES FALLBACK: Full year data for products missing quarter data
-- =========================================================================

full_year_with_weights AS (
    SELECT
        *,
        EXP(-0.023 * days_ago) AS time_weight
    FROM iqr_cleaned
),

full_year_ordered AS (
    SELECT
        product_id,
        warehouse_id,
        daily_margin,
        time_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
            ORDER BY daily_margin
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cum_weight,
        SUM(time_weight) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS total_weight,
        COUNT(*) OVER (
            PARTITION BY product_id, warehouse_id
        ) AS data_points
    FROM full_year_with_weights
),

full_year_percentiles AS (
    SELECT
        product_id,
        warehouse_id,
        MIN(CASE WHEN cum_weight >= total_weight * 0.10 THEN daily_margin END) AS MIN_BOUNDARY,
        MIN(CASE WHEN cum_weight >= total_weight * 0.50 THEN daily_margin END) AS MEDIAN_BM,
        MIN(CASE WHEN cum_weight >= total_weight * 0.98 THEN daily_margin END) AS MAX_BOUNDARY
    FROM full_year_ordered
    WHERE data_points >= 5
    GROUP BY product_id, warehouse_id
),

weighted_percentiles AS (
    SELECT
        COALESCE(qp.product_id,   fy.product_id)   AS product_id,
        COALESCE(qp.warehouse_id, fy.warehouse_id) AS warehouse_id,
        COALESCE(qp.MIN_BOUNDARY, fy.MIN_BOUNDARY) AS MIN_BOUNDARY,
        COALESCE(qp.MEDIAN_BM,    fy.MEDIAN_BM)    AS MEDIAN_BM,
        COALESCE(qp.MAX_BOUNDARY, fy.MAX_BOUNDARY) AS MAX_BOUNDARY
    FROM full_year_percentiles fy
    LEFT JOIN quarter_percentiles qp
        ON fy.product_id = qp.product_id
        AND fy.warehouse_id = qp.warehouse_id
),


-- =========================================================================
-- OPTIMAL TRACK: Last 120 days, GP-maximizing margin bin
-- =========================================================================

recent_data AS (
    SELECT
        *,
        EXP(-0.023 * days_ago) AS time_weight
    FROM iqr_cleaned
    WHERE days_ago <= 120
),

margin_bins AS (
    SELECT
        product_id,
        warehouse_id,
        ROUND(daily_margin, 2) AS margin_bin,
        SUM(nmv * time_weight)          AS weighted_nmv,
        SUM(gross_profit * time_weight) AS weighted_gp,
        COUNT(*)                        AS obs_count
    FROM recent_data
    GROUP BY product_id, warehouse_id, ROUND(daily_margin, 2)
),

smoothed_performance AS (
    SELECT
        product_id,
        warehouse_id,
        margin_bin,
        obs_count,
        AVG(weighted_nmv) OVER (
            PARTITION BY product_id, warehouse_id
            ORDER BY margin_bin
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS smooth_nmv,
        AVG(weighted_gp) OVER (
            PARTITION BY product_id, warehouse_id
            ORDER BY margin_bin
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS smooth_gp,
        SUM(obs_count) OVER (
            PARTITION BY product_id, warehouse_id
            ORDER BY margin_bin
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ) AS smooth_obs
    FROM margin_bins
),

optimal_margin AS (
    SELECT
        product_id,
        warehouse_id,
        margin_bin AS optimal_bm
    FROM smoothed_performance
    WHERE smooth_obs >= 2
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_id, warehouse_id
        ORDER BY smooth_gp DESC, smooth_nmv DESC
    ) = 1
)


-- =========================================================================
-- FINAL OUTPUT
-- =========================================================================

SELECT
    COALESCE(wp.product_id, om.product_id) AS product_id,
    COALESCE(wp.warehouse_id, om.warehouse_id) AS warehouse_id,
    om.optimal_bm,
    wp.MIN_BOUNDARY,
    wp.MAX_BOUNDARY,
    wp.MEDIAN_BM
FROM weighted_percentiles wp
FULL OUTER JOIN optimal_margin om
    ON wp.product_id = om.product_id
    AND wp.warehouse_id = om.warehouse_id
WHERE COALESCE(wp.MIN_BOUNDARY, om.optimal_bm) IS NOT NULL
'''

# =============================================================================
# 10. BRAND MARKET PERCENTILES QUERY
# =============================================================================
BRAND_MARKET_PERCENTILES_QUERY = f'''
with scrapped as (
WITH latest_per_sku AS (
    SELECT product_name_ar,
           MAX(created_at::date) AS max_date
    FROM materialized_views.raw_scraped_data
    WHERE created_at::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 4
    GROUP BY product_name_ar
),
main_prices AS (
    SELECT DISTINCT r.*
    FROM materialized_views.raw_scraped_data r
    JOIN latest_per_sku l
      ON r.product_name_ar = l.product_name_ar
     AND r.created_at::date = l.max_date
),
cleaned_data AS (
    SELECT query_id AS product_id,
           mp.product_name_ar,
           price AS comp_prices,
           CASE coalesce(region, area)
               WHEN 'Giza' THEN 'Cairo'
               WHEN 'Alex' THEN 'Alexandria'
               ELSE coalesce(region, area)
           END AS region,
           source_app,
           supplier
    FROM main_prices mp
    JOIN materialized_views.competitors_mapping_fixed cm 
      ON cm.match = mp.product_name_ar
),
matched_prices AS (
    SELECT *,
           comp_prices / BASIC_UNIT_COUNT AS final_price
    FROM (
        SELECT *,
               MIN(diff) OVER (PARTITION BY product_id, coalesce(supplier, source_app), region) AS min_diff
        FROM (
            SELECT cd.*,
                   wac1,
                   pup.PACKING_UNIT_id,
                   BASIC_UNIT_COUNT,
                   wac1 * BASIC_UNIT_COUNT AS pu_price,
                   ABS(comp_prices - pu_price) / pu_price AS diff
            FROM cleaned_data cd
            JOIN finance.all_cogs f 
              ON f.product_id = cd.product_id
             AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN from_date AND to_date
            JOIN PACKING_UNIT_PRODUCTs pup 
              ON pup.product_id = cd.product_id
             AND diff < 0.3
        )
        QUALIFY diff = min_diff
    )
),
target_regions AS (
    SELECT 'Cairo' AS region
    UNION ALL SELECT 'Giza'
    UNION ALL SELECT 'Alexandria'
    UNION ALL SELECT 'Delta East'
    UNION ALL SELECT 'Delta West'
    UNION ALL SELECT 'Upper Egypt'
),
product_suppliers AS (
    SELECT DISTINCT product_id, product_name_ar, coalesce(supplier, source_app) AS supplier_key,
           PACKING_UNIT_id, BASIC_UNIT_COUNT, wac1
    FROM matched_prices
),
all_combos AS (
    SELECT ps.*, tr.region AS target_region
    FROM product_suppliers ps
    CROSS JOIN target_regions tr
),
fallback_prices AS (
    SELECT ac.product_id,
           ac.product_name_ar,
           ac.supplier_key,
           ac.target_region,
           ac.PACKING_UNIT_id,
           ac.BASIC_UNIT_COUNT,
           ac.wac1,
           mp.region AS source_region,
           mp.comp_prices,
           mp.final_price,
           mp.source_app,
           mp.supplier,
           CASE
               WHEN mp.region = ac.target_region THEN 1
               WHEN ac.target_region = 'Giza' AND mp.region = 'Cairo' THEN 1
               WHEN mp.region = 'Cairo' AND ac.target_region NOT IN ('Cairo', 'Giza') THEN 2
               WHEN ac.target_region IN ('Delta East', 'Delta West', 'Alexandria')
                    AND mp.region IN ('Delta East', 'Delta West', 'Alexandria')
                    AND mp.region != ac.target_region THEN 3
               ELSE NULL
           END AS priority
    FROM all_combos ac
    JOIN matched_prices mp
      ON mp.product_id = ac.product_id
     AND coalesce(mp.supplier, mp.source_app) = ac.supplier_key
    WHERE CASE
              WHEN ac.target_region IN ('Cairo', 'Giza') 
                   THEN mp.region = 'Cairo'
              WHEN ac.target_region = 'Upper Egypt' 
                   THEN mp.region IN ('Upper Egypt', 'Cairo')
              WHEN ac.target_region IN ('Delta East', 'Delta West', 'Alexandria')
                   THEN mp.region IN ('Cairo', 'Delta East', 'Delta West', 'Alexandria')
              ELSE FALSE
          END
)

SELECT product_id,
       product_name_ar as sku,
       b.name_ar as brand,
       c.name_ar as cat,
       target_region AS region,
       source_region,
       fp.priority,
       comp_prices,
       final_price,
       source_app,
       supplier,
       supplier_key,
       PACKING_UNIT_id,
       BASIC_UNIT_COUNT,
       wac1
FROM fallback_prices fp 
join products p on p.id = fp.product_id
join brands b on b.id = p.brand_id
join categories c on c.id = p.category_id
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_id, supplier_key, target_region
    ORDER BY fp.priority ASC
) = 1
),

source_region_p50 AS (
    SELECT brand, cat, source_region,
           PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY final_price) AS p50_price
    FROM scrapped
    WHERE priority = 1
    GROUP BY brand, cat, source_region
),

scrapped_filtered AS (
    SELECT s.*
    FROM scrapped s
    LEFT JOIN source_region_p50 p
      ON p.brand = s.brand AND p.cat = s.cat AND p.source_region = s.source_region
    WHERE s.priority = 1
       OR s.final_price >= COALESCE(p.p50_price, 0)
),

bensoliman as (
WITH lower as (
    select distinct product_id, new_d*bs_price as ben_soliman_price, INJECTION_DATE
    from (
        select maxab_product_id as product_id, INJECTION_DATE, wac1, wac_p,
            (bs_price) as bs_price, diff, cu_price,
            case when p1 > 1 then child_quantity else 0 end as scheck,
            round(p1/2)*2 as p1, p2,
            case when (ROUND(p1 / scheck) * scheck) = 0 then p1 else (ROUND(p1 / scheck) * scheck) end as new_d
        from (
            select sm.*, wac1, wac_p, 
                abs((bs_price)-(wac_p*maxab_basic_unit_count))/(wac_p*maxab_basic_unit_count) as diff,
                cpc.price as cu_price, pup.child_quantity,
                round((cu_price/bs_price)) as p1, 
                round(((bs_price)/cu_price)) as p2
            from materialized_views.savvy_mapping sm 
            join finance.all_cogs f on f.product_id = sm.maxab_product_id 
                and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between f.from_Date and f.to_date
            join PACKING_UNIT_PRODUCTS pu on pu.product_id = sm.maxab_product_id and pu.IS_BASIC_UNIT = 1 
            join cohort_product_packing_units cpc on cpc.PRODUCT_PACKING_UNIT_ID = pu.id and cohort_id = 700 
            join packing_unit_products pup on pup.product_id = sm.maxab_product_id and pup.is_basic_unit = 1  
            where bs_price is not null 
                and INJECTION_DATE::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 5 
                and diff > 0.3 and p1 > 1
        )
    )
    qualify max(INJECTION_DATE) over(partition by product_id) = INJECTION_DATE
),

m_bs as (
    select z.* from (
        select maxab_product_id as product_id, avg(bs_final_price) as ben_soliman_price, INJECTION_DATE
        from (
            select *, row_number() over(partition by maxab_product_id order by diff) as rnk_2 
            from (
                select *, (bs_final_price-wac_p)/wac_p as diff_2 
                from (
                    select *, bs_price/maxab_basic_unit_count as bs_final_price 
                    from (
                        select *, row_number() over(partition by maxab_product_id, maxab_pu order by diff) as rnk 
                        from (
                            select *, max(INJECTION_DATE::date) over(partition by maxab_product_id, maxab_pu) as max_date
                            from (
                                select sm.*, wac1, wac_p, 
                                    abs(bs_price-(wac_p*maxab_basic_unit_count))/(wac_p*maxab_basic_unit_count) as diff 
                                from materialized_views.savvy_mapping sm 
                                join finance.all_cogs f on f.product_id = sm.maxab_product_id 
                                    and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between f.from_Date and f.to_date
                                where bs_price is not null 
                                    and INJECTION_DATE::date >= CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 5 
                                    and diff < 0.3
                            )
                            qualify max_date = INJECTION_DATE
                        ) qualify rnk = 1 
                    )
                ) where diff_2 between -0.5 and 0.5 
            ) qualify rnk_2 = 1 
        ) group by all
    ) z 
    join finance.all_cogs f on f.product_id = z.product_id 
        and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between f.from_Date and f.to_date
    where ben_soliman_price between f.wac_p*0.8 and f.wac_p*1.3
)
select product_id, avg(ben_soliman_price) as ben_soliman_price
from (
select product_id, ben_soliman_price, INJECTION_DATE
from (
    select * from (
        select *, 1 as prio from m_bs 
        union all
        select *, 2 as prio from lower
    )
    qualify max(INJECTION_DATE) over(partition by product_id) = INJECTION_DATE
)
qualify prio = min(prio) over(partition by product_id)
) x 
group by all

),

all_regions AS (
    SELECT 'Cairo' AS region
    UNION ALL SELECT 'Giza'
    UNION ALL SELECT 'Alexandria'
    UNION ALL SELECT 'Delta East'
    UNION ALL SELECT 'Delta West'
    UNION ALL SELECT 'Upper Egypt'
),

market_place as (
WITH seller_region AS (
    SELECT 
        seller_retailer.retailer_id,
        CASE WHEN regions.name_en = 'Greater Cairo' THEN cities.name_en ELSE regions.name_en END AS region,
        seller_id,
        seller_retailer.POLYGON_ID
    FROM MATERIALIZED_VIEWS.SELLERS_RETAILERS_MAPPING seller_retailer
    JOIN retailers ON retailers.id = seller_retailer.retailer_id
    JOIN materialized_views.retailer_polygon ON materialized_views.retailer_polygon.retailer_id = retailers.id
    JOIN districts ON districts.id = materialized_views.retailer_polygon.district_id
    JOIN cities ON cities.id = districts.city_id
    JOIN states ON states.id = cities.state_id
    JOIN regions ON regions.id = states.region_id
    JOIN egypt_marketplace.sellers ON sellers.id = seller_retailer.seller_id AND sellers.status = 'ACTIVE'
),

recent_price AS (
    SELECT
        wp.product_id,
        wp.packing_unit_id AS product_pu,
        wp.price,
        wp.max_per_order,
        warehouses.seller_id,
        warehouses.MIN_TICKET_SIZE
    FROM egypt_marketplace.warehouse_products wp
    LEFT JOIN egypt_marketplace.warehouses ON warehouses.id = wp.warehouse_id
    WHERE wp.AVAILABLE > 0 
      AND wp.total_stock > 0
      AND activation = 'true'
),

mp_raw AS (
    SELECT DISTINCT
        sr.region,
        rp.product_id,
        rp.product_pu,
        rp.price,
        rp.max_per_order,
        rp.seller_id,
        rp.MIN_TICKET_SIZE
    FROM recent_price rp
    JOIN seller_region sr ON sr.seller_id = rp.seller_id
),

pu_wacs AS (
    SELECT
        pup.product_id,
        pup.PACKING_UNIT_ID AS pu_id,
        pup.BASIC_UNIT_COUNT AS buc,
        f.wac1,
        f.wac4,
        f.wac_p,
        pup.BASIC_UNIT_COUNT * f.wac1 AS pu_wac1,
        pup.BASIC_UNIT_COUNT * f.wac4 AS pu_wac4
    FROM PACKING_UNIT_PRODUCTS pup
    LEFT JOIN finance.all_cogs f 
      ON f.product_id = pup.product_id
     AND CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) BETWEEN f.from_date AND f.to_date
),

mp_filtered AS (
    SELECT
        m.region,
        m.product_id,
        m.price,
        m.max_per_order,
        m.seller_id,
        m.MIN_TICKET_SIZE,
        pw.pu_id
    FROM mp_raw m
    JOIN pu_wacs pw ON pw.product_id = m.product_id
    WHERE pw.pu_wac4 > 0
      AND ROUND((m.price - pw.pu_wac4) / pw.pu_wac4 * 100, 2) BETWEEN -40 AND 40
),

ticket_iqr AS (
    SELECT *,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY MIN_TICKET_SIZE) 
            OVER (PARTITION BY region, product_id, pu_id) AS ts_q1,
        PERCENTILE_CONT(0.85) WITHIN GROUP (ORDER BY MIN_TICKET_SIZE) 
            OVER (PARTITION BY region, product_id, pu_id) AS ts_q3
    FROM mp_filtered
),
ticket_cleaned AS (
    SELECT *
    FROM ticket_iqr
    WHERE MIN_TICKET_SIZE >= ts_q1 - 1.5 * (ts_q3 - ts_q1)
      AND MIN_TICKET_SIZE <= ts_q3 + 3.0 * (ts_q3 - ts_q1)
),

order_iqr AS (
    SELECT *,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY max_per_order) 
            OVER (PARTITION BY region, product_id, pu_id) AS mo_q1,
        PERCENTILE_CONT(0.85) WITHIN GROUP (ORDER BY max_per_order) 
            OVER (PARTITION BY region, product_id, pu_id) AS mo_q3
    FROM ticket_cleaned
)

SELECT
    order_iqr.region,
    order_iqr.product_id,
    order_iqr.price,
    order_iqr.max_per_order,
    order_iqr.seller_id,
    order_iqr.MIN_TICKET_SIZE,
    order_iqr.pu_id,
    pup.basic_unit_count
FROM order_iqr
join packing_unit_products pup on pup.product_id = order_iqr.product_id and pup.packing_unit_id = order_iqr.pu_id
WHERE max_per_order >= mo_q1 - 1.5 * (mo_q3 - mo_q1)

),
collected_data as (
select *
from (
SELECT r.region, b.product_id, ben_soliman_price as final_price, 'ben' AS source
FROM bensoliman b
CROSS JOIN all_regions r

union all 

select region, product_id, final_price, source_app as source
from scrapped_filtered

union all 

select region, product_id, price/basic_unit_count as final_price, 'MP' as source
from market_place

) 
),
last_3_months_sales as (
    select case when regions.id = 2 then cities.name_en else regions.name_en end as region,
    pso.product_id,
    sum(pso.total_price) as nmv

    FROM product_sales_order pso
    JOIN sales_orders so ON so.id = pso.sales_order_id
    JOIN materialized_views.retailer_polygon on materialized_views.retailer_polygon.retailer_id=so.retailer_id
    JOIN districts on districts.id=materialized_views.retailer_polygon.district_id
    JOIN cities on cities.id=districts.city_id
    join states on states.id=cities.state_id
    join regions on regions.id=states.region_id

    WHERE so.created_at::date between CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 90 
          and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP())::date - 1
        AND so.sales_order_status_id not in (7,12)
        AND so.channel IN ('telesales','retailer')
        AND pso.purchased_item_count <> 0

    GROUP BY ALL
    ORDER BY 1
)

select region, brand, cat,
count(final_price) as num_rows,
sum(nmv) as nmv,
PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY margin) as perc_5,
PERCENTILE_CONT(0.15) WITHIN GROUP (ORDER BY margin) as perc_15,
PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY margin) as perc_25,
PERCENTILE_CONT(0.4) WITHIN GROUP (ORDER BY margin) as perc_40,
PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY margin) as perc_50,
PERCENTILE_CONT(0.6) WITHIN GROUP (ORDER BY margin) as perc_60,
PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY margin) as perc_75,
PERCENTILE_CONT(0.85) WITHIN GROUP (ORDER BY margin) as perc_85,
PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY margin) as perc_95
from(
select cd.*, b.name_ar as brand, c.name_ar as cat, wac1, wac_p,
    (final_price-wac_p)/final_price as margin, coalesce(nmv,0) as nmv
from collected_data cd 
left join last_3_months_sales ls on ls.product_id = cd.product_id and ls.region = cd.region
join products p on p.id = cd.product_id
join brands b on b.id = p.brand_id
join categories c on c.id = p.category_id
join finance.all_cogs f on f.product_id = cd.product_id 
    and CONVERT_TIMEZONE('{TIMEZONE}', 'Africa/Cairo', CURRENT_TIMESTAMP()) between from_date and to_date 
where final_price > 0.9*wac_p
)
group by all 
having num_rows > 2
'''

print("All queries defined ✓")


In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def price_analysis(row):
    """
    Analyze prices and calculate percentiles for a product.
    
    Collects prices from all sources (Ben Soliman, Marketplace, Scrapped),
    filters for valid prices within acceptable range, and calculates percentiles.
    """
    wac = row['wac_p']
    avg_margin = row['avg_margin'] if row['avg_margin'] >= 0.01 else row['target_margin']
    std = np.maximum(row['std'], 0.0025)
    target_margin = row['target_margin']
    max_marg = np.maximum(avg_margin, target_margin)
    
    # Collect all price points from different sources
    price_list = [
        row.get('ben_soliman_price'), 
        row.get('final_min_price'), 
        row.get('final_mod_price'),
        row.get('final_max_price'), 
        row.get('final_true_min'), 
        row.get('final_true_max'),
        row.get('min_scrapped'), 
        row.get('scrapped25'), 
        row.get('scrapped50'), 
        row.get('scrapped75'), 
        row.get('max_scrapped')
    ]
    
    # Filter valid prices within acceptable range
    valid_prices = sorted({
        x for x in price_list 
        if x and not pd.isna(x) and x != 0 
        and wac / (1 - (avg_margin - (10 * std))) <= x <= wac / (1 - (max_marg + 10 * std))
        and x >= wac * (0.9 + target_margin * 0.7)
    })
    
    if not valid_prices:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    return (
        np.min(valid_prices),
        np.percentile(valid_prices, 25),
        np.percentile(valid_prices, 50),
        np.percentile(valid_prices, 75),
        np.max(valid_prices)
    )


def calculate_step_bounds(row):
    """Calculate below/above market bounds based on price steps."""
    wac = row['wac_p']
    std = row['std']
    target_margin = row.get('target_margin', 0.05)
    
    prices = [
        row['minimum'], 
        row['percentile_25'], 
        row['percentile_50'], 
        row['percentile_75'], 
        row['maximum']
    ]
    
    # Calculate valid steps between price points
    valid_steps = []
    for i in range(len(prices) - 1):
        step = prices[i + 1] - prices[i]
        if (step / wac) <= std * 1.2:
            valid_steps.append(step)
    
    avg_step = np.mean(valid_steps) if valid_steps else min(2 * std, 0.2 * target_margin)
    
    new_min = prices[0] - avg_step if (prices[0] - avg_step) >= wac else prices[0]
    new_max = prices[-1] + avg_step if (prices[-1] + avg_step) >= wac else prices[-1]
    
    return new_min, new_max


def weighted_median(series, weights):
    """Calculate weighted median of a series."""
    valid = ~series.isna() & ~weights.isna()
    s = series[valid]
    w = weights[valid]
    if len(s) == 0:
        return np.nan
    order = np.argsort(s)
    s, w = s.iloc[order], w.iloc[order]
    return s.iloc[np.searchsorted(np.cumsum(w), w.sum() / 2)]


print("Helper functions defined ✓")


In [ ]:
# =============================================================================
# MAIN FUNCTION 1: get_market_data()
# =============================================================================
# This function fetches and processes all market price data from Snowflake
# NO INPUT REQUIRED - all data is fetched directly

def get_market_data() -> pd.DataFrame:
    """
    Fetch and process all market prices from Snowflake.
    
    NO INPUT REQUIRED - All data is fetched directly from Snowflake.
    
    Process:
    1. Fetch Ben Soliman, Marketplace, and Scrapped prices
    2. Outer join all price sources
    3. Add cohort IDs and supporting data (sales, margin stats, targets)
    4. Process group-level prices (weighted median)
    5. Apply price coverage filtering
    6. Calculate price percentiles
    7. Convert prices to margins
    
    Returns:
        DataFrame with columns:
        - cohort_id, product_id, region
        - Raw prices: ben_soliman_price, final_min_price, etc.
        - Percentiles: minimum, percentile_25, percentile_50, percentile_75, maximum
        - Margins: below_market, market_min, market_25, market_50, market_75, market_max, above_market
    """
    print("\n" + "="*70)
    print("FETCHING MARKET DATA")
    print("="*70)
    print(f"Timestamp: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time")
    
    # =========================================================================
    # Step 1: Fetch all raw price data
    # =========================================================================
    print("\nStep 1: Fetching raw price data...")
    
    print("  1.1 Ben Soliman prices...")
    df_ben_soliman = query_snowflake(BEN_SOLIMAN_QUERY)
    print(f"      Loaded {len(df_ben_soliman)} records")
    
    print("  1.2 Marketplace prices...")
    df_marketplace = query_snowflake(MARKETPLACE_PRICES_QUERY)
    df_marketplace['final_true_min'] = np.nan
    print(f"      Loaded {len(df_marketplace)} records")
    
    print("  1.3 Scrapped prices...")
    df_scrapped = query_snowflake(SCRAPPED_QUERY)
    print(f"      Loaded {len(df_scrapped)} records")
    
    print("  1.4 Product groups...")
    df_groups = query_snowflake(GROUPS_QUERY)
    print(f"      Loaded {len(df_groups)} records")
    
    print("  1.5 Sales data (for NMV weighting)...")
    df_sales = query_snowflake(SALES_QUERY)
    print(f"      Loaded {len(df_sales)} records")
    
    print("  1.6 Margin stats...")
    df_margin_stats = query_snowflake(MARGIN_STATS_QUERY)
    print(f"      Loaded {len(df_margin_stats)} records")
    
    print("  1.7 Target margins...")
    df_targets = query_snowflake(TARGET_MARGINS_QUERY)
    print(f"      Loaded {len(df_targets)} records")
    
    print("  1.8 Product base (WAC)...")
    df_product_base = query_snowflake(PRODUCT_BASE_QUERY)
    print(f"      Loaded {len(df_product_base)} records")
    
    # =========================================================================
    # Step 2: Outer join all market price sources
    # =========================================================================
    print("\nStep 2: Joining all market price sources (outer join)...")
    
    # Start with marketplace prices (has region + product_id)
    market_data = df_marketplace.copy()
    
    # Outer join with scrapped data (by region + product_id)
    market_data = market_data.merge(df_scrapped, on=['region', 'product_id'], how='outer')
    
    # Outer join with Ben Soliman prices (by product_id only - expand to all regions)
    all_regions = pd.DataFrame({'region': ['Cairo', 'Giza', 'Delta West', 'Delta East', 'Upper Egypt', 'Alexandria']})
    df_ben_soliman_expanded = df_ben_soliman.merge(all_regions, how='cross')
    
    # Outer join with Ben Soliman
    market_data = market_data.merge(df_ben_soliman_expanded, on=['region', 'product_id'], how='outer')
    
    print(f"    Market prices base: {len(market_data)} records")
    
    # =========================================================================
    # Step 3: Add cohort IDs and supporting data
    # =========================================================================
    print("\nStep 3: Adding cohort IDs and supporting data...")
    
    market_data = market_data.merge(REGION_COHORT_DF, on='region')
    
    # Add sales data for NMV weighting
    market_data = market_data.merge(
        df_sales[['cohort_id', 'product_id', 'nmv', 'sku', 'brand', 'cat']], 
        on=['cohort_id', 'product_id'], 
        how='left'
    )
    market_data['nmv'] = market_data['nmv'].fillna(0)
    
    # Merge product groups
    market_data = market_data.merge(df_groups, on='product_id', how='left')
    
    # Remove duplicates
    market_data = market_data.drop_duplicates(subset=['cohort_id', 'product_id'])
    
    print(f"    Records after adding cohorts: {len(market_data)}")
    
    # =========================================================================
    # Step 4: Group-level price processing
    # =========================================================================
    print("\nStep 4: Processing group-level prices...")
    
    price_cols = [
        'ben_soliman_price', 'final_min_price', 'final_max_price', 'final_mod_price', 
        'final_true_min', 'final_true_max', 'min_scrapped', 'scrapped25', 
        'scrapped50', 'scrapped75', 'max_scrapped'
    ]
    
    groups_data = market_data[~market_data['group_id'].isna()].copy()
    
    if len(groups_data) > 0:
        groups_data['group_nmv'] = groups_data.groupby(['group_id', 'cohort_id'])['nmv'].transform('sum')
        groups_data['cntrb'] = (groups_data['nmv'] / groups_data['group_nmv']).fillna(1)
        
        # Flag if any price column is non-NaN
        groups_data['flag_non_nan'] = groups_data[price_cols].notna().any(axis=1).astype(int)
        
        # Perform weighted aggregation
        groups_agg = (
            groups_data[groups_data['flag_non_nan'] == 1]
            .groupby(['group_id', 'cohort_id'])
            .apply(lambda g: pd.Series({
                col: weighted_median(g[col], g['cntrb']) for col in price_cols
            }))
            .reset_index()
        )
        
        # Fill missing prices with group-level prices
        merged = market_data.merge(groups_agg, on=['group_id', 'cohort_id'], how='left', suffixes=('', '_group'))
        for col in price_cols:
            if f'{col}_group' in merged.columns:
                merged[col] = merged[col].fillna(merged[f'{col}_group'])
        
        market_data = merged.drop(columns=[f'{c}_group' for c in price_cols if f'{c}_group' in merged.columns], errors='ignore')
        
        # Add missing group SKUs
        missing_groups_skus = df_groups.merge(groups_agg, on='group_id')
        missing_groups_skus = missing_groups_skus.merge(REGION_COHORT_DF, on='cohort_id')
        market_data = pd.concat([market_data, missing_groups_skus])
        market_data = market_data.drop_duplicates(subset=['cohort_id', 'product_id'], keep='first')
    
    print(f"    Records after group processing: {len(market_data)}")
    
    # =========================================================================
    # Step 5: Add WAC and margin data
    # =========================================================================
    print("\nStep 5: Adding WAC and margin data...")
    
    # Drop nmv and re-merge sales
    market_data = market_data.drop(columns=['nmv'], errors='ignore')
    market_data = market_data.merge(
        df_sales[['cohort_id', 'product_id', 'nmv']], 
        on=['cohort_id', 'product_id'], 
        how='left'
    )
    
    # Add WAC from product base
    market_data = market_data.merge(
        df_product_base[['cohort_id', 'product_id', 'wac_p', 'brand', 'cat']].drop_duplicates(), 
        on=['cohort_id', 'product_id'], 
        how='left',
        suffixes=('', '_base')
    )
    # Fill brand/cat from base if missing
    if 'brand_base' in market_data.columns:
        market_data['brand'] = market_data['brand'].fillna(market_data['brand_base'])
        market_data['cat'] = market_data['cat'].fillna(market_data['cat_base'])
        market_data = market_data.drop(columns=['brand_base', 'cat_base'], errors='ignore')
    
    # Add margin stats
    market_data = market_data.merge(df_margin_stats, on=['cohort_id', 'product_id'], how='left')
    
    # Add target margins
    market_data = market_data.merge(df_targets, on=['brand', 'cat'], how='left')
    market_data['target_margin'] = market_data['target_bm'].fillna(market_data['cat_target_margin']).fillna(0)
    market_data = market_data.drop(columns=['target_bm', 'cat_target_margin'], errors='ignore')
    
    # Fill NaN values with defaults
    market_data['std'] = market_data['std'].fillna(0.01)
    market_data['avg_margin'] = market_data['avg_margin'].fillna(0)
    
    # Filter out records without WAC
    market_data = market_data[~market_data['wac_p'].isna()]
    
    print(f"    Records with WAC: {len(market_data)}")
    
    # =========================================================================
    # Step 6: Price coverage filtering
    # =========================================================================
    print("\nStep 6: Filtering by price coverage...")
    
    market_data['ben'] = 0
    market_data['MP'] = 0
    market_data['sp'] = 0
    
    # Ben Soliman: 1 point if present
    market_data.loc[~market_data['ben_soliman_price'].isna(), 'ben'] = 1
    
    # Marketplace: 1 point if single price, 3 points if range
    market_data.loc[(market_data['final_min_price'] == market_data['final_max_price']) & 
                    (~market_data['final_min_price'].isna()), 'MP'] = 1
    market_data.loc[(market_data['final_min_price'] != market_data['final_max_price']) & 
                    (~market_data['final_min_price'].isna()), 'MP'] = 3
    
    # Scrapped: 1 point if single price, 5 points if range
    market_data.loc[(market_data['min_scrapped'] == market_data['max_scrapped']) & 
                    (~market_data['min_scrapped'].isna()), 'sp'] = 1
    market_data.loc[(market_data['min_scrapped'] != market_data['max_scrapped']) & 
                    (~market_data['min_scrapped'].isna()), 'sp'] = 5
    
    market_data['total_p'] = market_data['ben'] + market_data['MP'] + market_data['sp']
    
    # Filter: keep only products with total_p > 2
    market_data = market_data[market_data['total_p'] >= 2]
    
    print(f"    Records after price coverage filter: {len(market_data)}")
    
    # =========================================================================
    # Step 7: Apply price analysis
    # =========================================================================
    print("\nStep 7: Calculating price percentiles...")
    
    market_data[['minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum']] = \
        market_data.apply(price_analysis, axis=1, result_type='expand')
    
    # Filter out records without valid price analysis
    market_data = market_data[~market_data['minimum'].isna()]
    
    # Calculate below/above market bounds
    market_data[['below_market', 'above_market']] = market_data.apply(calculate_step_bounds, axis=1, result_type='expand')
    
    print(f"    Records after price analysis: {len(market_data)}")
    
    # =========================================================================
    # Step 8: Convert prices to margins
    # =========================================================================
    print("\nStep 8: Converting prices to margins...")
    
    market_data['below_market'] = (market_data['below_market'] - market_data['wac_p']) / market_data['below_market']
    market_data['market_min'] = (market_data['minimum'] - market_data['wac_p']) / market_data['minimum']
    market_data['market_25'] = (market_data['percentile_25'] - market_data['wac_p']) / market_data['percentile_25']
    market_data['market_50'] = (market_data['percentile_50'] - market_data['wac_p']) / market_data['percentile_50']
    market_data['market_75'] = (market_data['percentile_75'] - market_data['wac_p']) / market_data['percentile_75']
    market_data['market_max'] = (market_data['maximum'] - market_data['wac_p']) / market_data['maximum']
    market_data['above_market'] = (market_data['above_market'] - market_data['wac_p']) / market_data['above_market']
    
    # =========================================================================
    # Step 9: Select output columns
    # =========================================================================
    market_columns = [
        'cohort_id', 'product_id', 'region',
        # Raw prices
        'ben_soliman_price', 
        'final_min_price', 'final_max_price', 'final_mod_price', 'final_true_min', 'final_true_max',
        'min_scrapped', 'scrapped25', 'scrapped50', 'scrapped75', 'max_scrapped',
        # Price Percentiles
        'minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum',
        # Margin Tiers
        'below_market', 'market_min', 'market_25', 'market_50', 'market_75', 'market_max', 'above_market'
    ]
    market_data = market_data[[c for c in market_columns if c in market_data.columns]]
    
    # =========================================================================
    # Summary
    # =========================================================================
    print("\n" + "="*70)
    print("MARKET DATA COMPLETE")
    print("="*70)
    print(f"Total records: {len(market_data)}")
    print(f"  - With marketplace prices: {(~market_data['final_min_price'].isna()).sum()}")
    print(f"  - With scrapped prices: {(~market_data['min_scrapped'].isna()).sum()}")
    print(f"  - With Ben Soliman prices: {(~market_data['ben_soliman_price'].isna()).sum()}")
    
    return market_data


print("get_market_data() function defined ✓")


In [ ]:
# =============================================================================
# MAIN FUNCTION 2: get_margin_tiers()
# =============================================================================
# This function fetches margin boundaries and calculates margin tiers
# NO INPUT REQUIRED - all data is fetched directly

def get_margin_tiers() -> pd.DataFrame:
    """
    Compute margin boundaries and calculate margin tiers from Snowflake.
    
    NO INPUT REQUIRED - All data is fetched directly from Snowflake.
    
    Process:
    1. Compute margin boundaries from sales data (IQR-cleaned, seasonally
       weighted percentiles for min/max/median, GP-maximizing for optimal)
    2. Map warehouse_id to cohort_id and region
    3. Calculate 8 margin tiers:
       - margin_tier_below: effective minimum
       - margin_tier_1 to margin_tier_5: Within range
       - margin_tier_above_1, margin_tier_above_2: Above maximum
    
    Returns:
        DataFrame with columns:
        - product_id, warehouse_id, region, cohort_id
        - optimal_bm, min_boundary, max_boundary, median_bm
        - effective_min_margin, margin_step
        - margin_tier_below, margin_tier_1, margin_tier_2, margin_tier_3,
          margin_tier_4, margin_tier_5, margin_tier_above_1, margin_tier_above_2
    """
    print("\n" + "="*70)
    print("FETCHING MARGIN TIERS")
    print("="*70)
    print(f"Timestamp: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time")
    
    # =========================================================================
    # Step 1: Compute margin boundaries from sales data
    # =========================================================================
    print("\nStep 1: Computing margin boundaries from sales data...")
    
    df_margin_boundaries = query_snowflake(MARGIN_BOUNDARIES_QUERY)
    print(f"    Loaded {len(df_margin_boundaries)} records (per product per warehouse)")
    
    if len(df_margin_boundaries) == 0:
        print("    Warning: No margin boundaries found!")
        return pd.DataFrame()
    
    # =========================================================================
    # Step 2: Map warehouse_id to cohort_id and region
    # =========================================================================
    print("\nStep 2: Mapping warehouses to cohorts...")
    
    df = df_margin_boundaries.copy()
    df = df.drop_duplicates(subset=['product_id', 'warehouse_id'])
    print(f"    Records after cohort mapping: {len(df)}")
    
    # =========================================================================
    # Step 3: Calculate margin tiers
    # =========================================================================
    print("\nStep 3: Calculating margin tiers...")
    
    df['effective_min_margin'] = df[['min_boundary', 'optimal_bm']].min(axis=1)
    df['margin_step'] = (df['max_boundary'] - df['effective_min_margin']) / 4
    
    df['margin_tier_below'] = df['effective_min_margin'] 
    df['margin_tier_1'] = df['effective_min_margin'] + (0.5 * df['margin_step'])
    df['margin_tier_2'] = df['effective_min_margin'] + df['margin_step']
    df['margin_tier_3'] = df['effective_min_margin'] + 2 * df['margin_step']
    df['margin_tier_4'] = df['effective_min_margin'] + 3 * df['margin_step']
    df['margin_tier_5'] = df['max_boundary']
    df['margin_tier_above_1'] = df['max_boundary'] + df['margin_step']
    df['margin_tier_above_2'] = df['max_boundary'] + 2 * df['margin_step']
    
    # =========================================================================
    # Step 4: Select output columns
    # =========================================================================
    output_cols = [
        'product_id', 'warehouse_id',
        'optimal_bm', 'min_boundary', 'max_boundary', 'median_bm',
        'effective_min_margin', 'margin_step',
        'margin_tier_below', 'margin_tier_1', 'margin_tier_2', 'margin_tier_3',
        'margin_tier_4', 'margin_tier_5', 'margin_tier_above_1', 'margin_tier_above_2'
    ]
    df = df[[c for c in output_cols if c in df.columns]]
    
    # =========================================================================
    # Summary
    # =========================================================================
    print("\n" + "="*70)
    print("MARGIN TIERS COMPLETE")
    print("="*70)
    print(f"Total records: {len(df)}")
    print(f"\nMargin Tier Structure:")
    print(f"  margin_tier_below:   effective_min_margin")
    print(f"  margin_tier_1:       effective_min + 0.5*step")
    print(f"  margin_tier_2:       effective_min + 1*step")
    print(f"  margin_tier_3:       effective_min + 2*step")
    print(f"  margin_tier_4:       effective_min + 3*step")
    print(f"  margin_tier_5:       max_boundary")
    print(f"  margin_tier_above_1: max_boundary + 1*step")
    print(f"  margin_tier_above_2: max_boundary + 2*step")
    
    return df


print("get_margin_tiers() function defined ✓")


In [1]:
# =============================================================================
# MAIN FUNCTION 3: get_brand_market_percentiles()
# =============================================================================

def get_brand_market_percentiles() -> pd.DataFrame:
    """
    Fetch brand-level market margin percentiles aggregated by region, brand, category.
    
    Collects prices from all 3 sources (scrapped, ben soliman, marketplace),
    computes margin = (final_price - wac_p) / final_price, then aggregates
    per (region, brand, cat) into margin percentiles.
    
    Returns:
        DataFrame with columns: region, brand, cat, num_rows, nmv,
        perc_5, perc_15, perc_25, perc_40, perc_50, perc_60, perc_75, perc_85, perc_95
    """
    print("\n" + "="*70)
    print("FETCHING BRAND MARKET PERCENTILES")
    print("="*70)
    print(f"Timestamp: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo time")
    
    df_brand = query_snowflake(BRAND_MARKET_PERCENTILES_QUERY)
    print(f"  Loaded {len(df_brand)} brand-region-category records")
    
    if len(df_brand) == 0:
        print("  Warning: No brand market percentiles found!")
        return pd.DataFrame()
    
    for col in ['perc_5','perc_15','perc_25','perc_40','perc_50','perc_60','perc_75','perc_85','perc_95']:
        if col in df_brand.columns:
            df_brand[col] = pd.to_numeric(df_brand[col], errors='coerce')
    
    print(f"  Unique brands: {df_brand['brand'].nunique()}")
    print(f"  Unique categories: {df_brand['cat'].nunique()}")
    print(f"  Unique regions: {df_brand['region'].nunique()}")
    print("="*70)
    
    return df_brand


def fill_brand_market_fallback(df: pd.DataFrame, brand_percs: pd.DataFrame) -> pd.DataFrame:
    """
    Fill NaN market columns using brand-level percentiles for SKUs without SKU-level market data.
    
    For rows where 'minimum' is NaN (no SKU-level market data), looks up the row's
    brand + cat + region in brand_percs and fills market margin and price columns.
    
    Mapping:
        perc_5  -> below_market
        perc_15 -> market_min   / minimum
        perc_25 -> market_25    / percentile_25
        perc_50 -> market_50    / percentile_50
        perc_75 -> market_75    / percentile_75
        perc_95 -> market_max   / maximum
        perc_95 + (perc_95 - perc_75) -> above_market
    
    Adds 'market_data_source' column: 'sku', 'brand', or None.
    """
    if 'minimum' not in df.columns:
        df['minimum'] = np.nan
    
    if len(brand_percs) == 0:
        df['market_data_source'] = np.where(df['minimum'].notna(), 'sku', None)
        print("  Brand percentiles empty - no fallback applied")
        return df
    
    df = df.copy()
    
    margin_cols = ['below_market', 'market_min', 'market_25', 'market_50', 
                    'market_75', 'market_max', 'above_market']
    price_cols = ['minimum', 'percentile_25', 'percentile_50', 'percentile_75', 'maximum']
    for col in margin_cols + price_cols:
        if col not in df.columns:
            df[col] = np.nan
    
    has_sku_market = df['minimum'].notna()
    df['market_data_source'] = np.where(has_sku_market, 'sku', None)
    
    no_market_mask = ~has_sku_market
    no_market_count = no_market_mask.sum()
    print(f"\n  Brand fallback: {no_market_count} rows without SKU-level market data")
    
    if no_market_count == 0:
        print("  All rows have SKU-level market data - no fallback needed")
        return df
    
    brand_percs_clean = brand_percs[['region', 'brand', 'cat', 
                                      'perc_5', 'perc_15', 'perc_25', 'perc_50', 
                                      'perc_75', 'perc_95']].copy()
    brand_percs_clean = brand_percs_clean.drop_duplicates(subset=['region', 'brand', 'cat'])
    
    # Preserve original df indices through the merge
    df_no_market = df.loc[no_market_mask, ['brand', 'cat', 'region', 'wac_p']].copy()
    df_no_market['_orig_idx'] = df_no_market.index
    df_no_market = df_no_market.merge(brand_percs_clean, on=['region', 'brand', 'cat'], how='left')
    
    has_brand = df_no_market['perc_15'].notna()
    filled_count = has_brand.sum()
    print(f"  Brand fallback: matched {filled_count} rows to brand percentiles")
    
    if filled_count > 0:
        matched = df_no_market[has_brand]
        orig_idx = matched['_orig_idx'].values
        
        wac = matched['wac_p'].values.astype(float)
        p5  = matched['perc_5'].values.astype(float)
        p15 = matched['perc_15'].values.astype(float)
        p25 = matched['perc_25'].values.astype(float)
        p50 = matched['perc_50'].values.astype(float)
        p75 = matched['perc_75'].values.astype(float)
        p95 = matched['perc_95'].values.astype(float)
        
        df.loc[orig_idx, 'below_market'] = p5
        df.loc[orig_idx, 'market_min']   = p15
        df.loc[orig_idx, 'market_25']    = p25
        df.loc[orig_idx, 'market_50']    = p50
        df.loc[orig_idx, 'market_75']    = p75
        df.loc[orig_idx, 'market_max']   = p95
        df.loc[orig_idx, 'above_market'] = p95 + (p95 - p75)
        
        denom_15 = np.where(p15 < 1, 1 - p15, np.nan)
        denom_25 = np.where(p25 < 1, 1 - p25, np.nan)
        denom_50 = np.where(p50 < 1, 1 - p50, np.nan)
        denom_75 = np.where(p75 < 1, 1 - p75, np.nan)
        denom_95 = np.where(p95 < 1, 1 - p95, np.nan)
        
        df.loc[orig_idx, 'minimum']       = wac / denom_15
        df.loc[orig_idx, 'percentile_25'] = wac / denom_25
        df.loc[orig_idx, 'percentile_50'] = wac / denom_50
        df.loc[orig_idx, 'percentile_75'] = wac / denom_75
        df.loc[orig_idx, 'maximum']       = wac / denom_95
        
        df.loc[orig_idx, 'market_data_source'] = 'brand'
    
    remaining = no_market_count - filled_count
    print(f"  Brand fallback: {remaining} rows still without any market data")
    
    return df


print("get_brand_market_percentiles() function defined ✓")
print("fill_brand_market_fallback() function defined ✓")

NameError: name 'pd' is not defined

In [ ]:
# =============================================================================
# MODULE READY
# =============================================================================

print("\n" + "="*70)
print("MARKET DATA MODULE READY")
print("="*70)
print("\nAvailable functions (NO INPUT REQUIRED):")
print("  - get_market_data()              : Fetch and process all market prices")
print("  - get_margin_tiers()             : Fetch and calculate margin tiers")
print("  - get_brand_market_percentiles() : Fetch brand-level market margin percentiles")
print("  - fill_brand_market_fallback()   : Fill NaN market cols with brand percentiles")
print("\nUsage:")
print("  %run market_data_module.ipynb")
print("  df_market = get_market_data()")
print("  df_tiers = get_margin_tiers()")
print("  df_brand_percs = get_brand_market_percentiles()")
print("  df = fill_brand_market_fallback(df, df_brand_percs)")
print("="*70)


In [ ]:
# This cell intentionally left empty - old combined function removed
# Use get_market_data() and get_margin_tiers() instead
